# 05 - XGBoost + Hyperparameter Tuning

**Goal:** Learn gradient boosting, proper time-series cross-validation, and systematic hyperparameter tuning.

**Gradient boosting** builds trees sequentially — each tree tries to correct the errors of the previous one. XGBoost is an optimized implementation with regularization to prevent overfitting.

Key difference from Random Forest:
- RF: trees are independent, averaged → reduces *variance*
- XGBoost: trees are sequential, each corrects prior errors → reduces *bias*

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'prediction'))

from prediction.src.evaluation import evaluate_model, plot_predictions_vs_actual, plot_residuals

DATA_DIR = os.path.join(PROJECT_ROOT, 'prediction', 'data')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

df = pd.read_parquet(os.path.join(DATA_DIR, 'features_all.parquet'))
with open(os.path.join(DATA_DIR, 'feature_columns.json')) as f:
    FEATURE_COLS = json.load(f)

TARGET = 'total_points'

max_gw = df['round'].max()
split_gw = int(max_gw * 0.75)
train = df[df['round'] <= split_gw]
test = df[df['round'] > split_gw]
X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_test, y_test = test[FEATURE_COLS], test[TARGET]

# Load prior results
prev_results = pd.read_csv(os.path.join(DATA_DIR, 'results_phase4.csv'), index_col=0)
results = prev_results.to_dict('index')

print(f'Train: {len(train):,} rows | Test: {len(test):,} rows')

## 1. Default XGBoost

First, see how XGBoost performs out of the box. Default hyperparameters are often surprisingly good.

In [ ]:
xgb_default = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_default.fit(X_train, y_train)

xgb_pred_train = xgb_default.predict(X_train)
xgb_pred_test = xgb_default.predict(X_test)

xgb_train = evaluate_model(y_train, xgb_pred_train)
xgb_test = evaluate_model(y_test, xgb_pred_test)
results['XGBoost (default)'] = xgb_test

print('XGBoost (default hyperparameters):')
print(f'  Train — MAE: {xgb_train["MAE"]:.3f}, R2: {xgb_train["R2"]:.3f}')
print(f'  Test  — MAE: {xgb_test["MAE"]:.3f}, R2: {xgb_test["R2"]:.3f}')

In [ ]:
fig = plot_predictions_vs_actual(y_test, xgb_pred_test, 'XGBoost (Default): Predicted vs Actual')
plt.show()

## 2. Time-Series Cross-Validation

Regular k-fold CV randomly mixes timepoints — a no-go for time series. `TimeSeriesSplit` creates expanding windows:

```
Fold 1: train=[GW1-5]     val=[GW6-10]
Fold 2: train=[GW1-10]    val=[GW11-15]
Fold 3: train=[GW1-15]    val=[GW16-20]
...
```

Each fold only looks forward, never backward. This simulates real prediction.

In [ ]:
# Visualize the splits
tscv = TimeSeriesSplit(n_splits=5)

# Sort train by round to ensure temporal ordering
train_sorted = train.sort_values('round').reset_index(drop=True)

print('TimeSeriesSplit folds:')
for i, (train_idx, val_idx) in enumerate(tscv.split(train_sorted)):
    train_gws = train_sorted.iloc[train_idx]['round']
    val_gws = train_sorted.iloc[val_idx]['round']
    print(f'  Fold {i+1}: train GW {train_gws.min()}-{train_gws.max()} '
          f'({len(train_idx):,} rows) | val GW {val_gws.min()}-{val_gws.max()} '
          f'({len(val_idx):,} rows)')

## 3. Hyperparameter Tuning with RandomizedSearchCV

XGBoost has many hyperparameters. The key ones:

| Parameter | What it controls | Too low → | Too high → |
|-----------|-----------------|-----------|------------|
| `n_estimators` | Number of trees | Underfitting | Slow, diminishing returns |
| `max_depth` | Tree complexity | Underfitting | Overfitting |
| `learning_rate` | Step size per tree | Needs more trees | Overfitting |
| `subsample` | Row sampling per tree | Underfitting | Overfitting |
| `colsample_bytree` | Feature sampling per tree | Underfitting | Overfitting |
| `min_child_weight` | Minimum leaf size | Overfitting | Underfitting |

`RandomizedSearchCV` samples random combinations instead of trying every one (which would be too slow).

In [ ]:
param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 10],
}

xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=50,
    cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_mean_absolute_error',
    random_state=42,
    verbose=1,
    n_jobs=1  # XGBoost already uses multiple cores internally
)

# Need to sort by time for TimeSeriesSplit to work correctly
train_sorted = train.sort_values('round')
xgb_search.fit(train_sorted[FEATURE_COLS], train_sorted[TARGET])

print(f'\nBest parameters:')
for k, v in xgb_search.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nBest CV MAE: {-xgb_search.best_score_:.3f}')

In [ ]:
# Evaluate best model on test set
best_xgb = xgb_search.best_estimator_

best_pred_train = best_xgb.predict(X_train)
best_pred_test = best_xgb.predict(X_test)

best_train = evaluate_model(y_train, best_pred_train)
best_test = evaluate_model(y_test, best_pred_test)
results['XGBoost (tuned)'] = best_test

print('XGBoost (tuned):')
print(f'  Train — MAE: {best_train["MAE"]:.3f}, R2: {best_train["R2"]:.3f}')
print(f'  Test  — MAE: {best_test["MAE"]:.3f}, R2: {best_test["R2"]:.3f}')

## 4. Early Stopping

Early stopping monitors validation error during training and stops when it hasn't improved for N rounds. This prevents overfitting without needing to guess `n_estimators`.

In [ ]:
# Use the best hyperparameters but with high n_estimators and early stopping
best_params = xgb_search.best_params_.copy()
best_params['n_estimators'] = 1000  # Set high, early stopping will find the right number

xgb_es = XGBRegressor(**best_params, random_state=42, n_jobs=-1, early_stopping_rounds=20)
xgb_es.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

es_pred = xgb_es.predict(X_test)
es_metrics = evaluate_model(y_test, es_pred)
results['XGBoost (early stopping)'] = es_metrics

print(f'Early stopping at {xgb_es.best_iteration} trees (out of max 1000)')
print(f'Test — MAE: {es_metrics["MAE"]:.3f}, R2: {es_metrics["R2"]:.3f}')

## 5. Sliding Window Experiment

Does training on recent data only (last 10 GWs) beat training on all history? Recent data is more relevant, but less data means more variance.

In [ ]:
window_sizes = [5, 10, 15, 20, split_gw]  # last N GWs, plus "all"
window_results = []

for w in window_sizes:
    cutoff = split_gw - w
    train_window = train[train['round'] > cutoff]
    
    if len(train_window) < 100:
        continue
    
    xgb_w = XGBRegressor(**xgb_search.best_params_, random_state=42, n_jobs=-1)
    xgb_w.fit(train_window[FEATURE_COLS], train_window[TARGET])
    pred = xgb_w.predict(X_test)
    m = evaluate_model(y_test, pred)
    
    label = f'Last {w} GWs' if w < split_gw else 'All'
    window_results.append({'window': label, 'train_rows': len(train_window), **m})

window_df = pd.DataFrame(window_results)
print('Sliding Window Results:')
print(window_df.round(3).to_string(index=False))
print('\nMore data usually wins unless there is significant distribution shift across the season.')

## 6. SHAP Values (Interpretability)

SHAP (SHapley Additive exPlanations) shows how much each feature contributed to each individual prediction. Unlike feature importance (which is global), SHAP explains *why* a specific prediction was high or low.

In [ ]:
try:
    import shap
    
    # Use a sample for speed
    sample_size = min(500, len(X_test))
    X_sample = X_test.sample(sample_size, random_state=42)
    
    explainer = shap.TreeExplainer(best_xgb)
    shap_values = explainer.shap_values(X_sample)
    
    # Summary plot — shows feature impact direction and magnitude
    plt.figure(figsize=(10, 10))
    shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
    plt.title('SHAP Summary: Top 20 Features')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print('SHAP not installed. Run: pip install shap')
except Exception as e:
    print(f'SHAP failed: {e}')
    print('This is optional — skip and proceed.')

## 7. Per-Position XGBoost

In [ ]:
print(f'{"Position":<15s} {"MAE":>8s} {"RMSE":>8s} {"R2":>8s} {"n_train":>8s} {"n_test":>8s}')
print('-' * 60)

pos_models = {}
for pos in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
    mask_train = train['Position'] == pos
    mask_test = test['Position'] == pos
    
    if mask_test.sum() == 0 or mask_train.sum() == 0:
        continue
    
    xgb_pos = XGBRegressor(**xgb_search.best_params_, random_state=42, n_jobs=-1)
    xgb_pos.fit(X_train[mask_train], y_train[mask_train])
    pred = xgb_pos.predict(X_test[mask_test])
    m = evaluate_model(y_test[mask_test], pred)
    pos_models[pos] = xgb_pos
    
    print(f'{pos:<15s} {m["MAE"]:8.3f} {m["RMSE"]:8.3f} {m["R2"]:8.3f} '
          f'{mask_train.sum():8d} {mask_test.sum():8d}')

## 8. Results Summary

In [ ]:
results_df = pd.DataFrame(results).T
results_df.index.name = 'Model'
results_df = results_df.sort_values('MAE')

print('All Models Comparison (sorted by MAE):')
print(results_df.round(3).to_string())

results_df.to_csv(os.path.join(DATA_DIR, 'results_phase5.csv'))

# Save best params for Phase 6
with open(os.path.join(DATA_DIR, 'best_xgb_params.json'), 'w') as f:
    json.dump(xgb_search.best_params_, f, indent=2)

print(f'\nResults saved. Best XGBoost params saved to best_xgb_params.json')

## Summary

**Key learnings:**
1. **Gradient boosting** — sequential trees that correct prior errors
2. **TimeSeriesSplit** — proper CV that respects temporal ordering
3. **RandomizedSearchCV** — efficient hyperparameter search
4. **Early stopping** — automatic n_estimators selection
5. **Sliding window** — recency vs volume tradeoff
6. **SHAP** — per-prediction explanations, not just global importance

Next: `06_model_comparison.ipynb` — systematic comparison and final model selection.